<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourseH2/blob/main/Session3/03_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks
**Environmental Statistics and Artificial Intelligence — Practical Block 3, Part 3**

This is the final method of the course — and the one behind today's AI boom. We'll build neural
networks from the ground up: by the end you'll understand *exactly* what a
network is, why it works, and how to build one in a few lines of code.

The good news: **you already know most of it.** A neural network is built from a piece you've used
all day — a regression — with one small twist.

What we'll cover:
1. **The single neuron**
2. **Activation functions**
3. **From neuron to network**
4. **How a network learns**
5. **Building real networks in Keras**
6. **Overfitting and how to stop it**

> *Note* neural networks are still **supervised learning** (we have inputs and
> answers). What's new is *flexibility* — they can learn almost any relationship, which makes them
> powerful for images, text, and sequences. The cost: they need more data and care than the simple
> models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 90
print("TensorFlow version:", tf.__version__)

> **Note:** the first time you import TensorFlow, it can take ~20
> seconds — that's normal. TensorFlow is the big engine that does the heavy maths; **Keras** is the
> interface we actually write, and it lives inside TensorFlow as `tf.keras`.

---
## 1. The single neuron — you already built one this morning

Remember multiple regression? It predicts a number as a weighted sum of inputs plus a constant:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$$

A **neuron** is *exactly this*, with new names and **one extra step** at the end:

$$\text{output} = f(\,w_1 x_1 + w_2 x_2 + \dots + b\,)$$

- the **weights** $w$ are just the regression coefficients $\beta$,
- the **bias** $b$ is just the intercept $\beta_0$,
- and $f$ is a new ingredient called the **activation function**

So a neuron = your regression, wrapped in a function $f$. That's the whole idea. Let's prove a single
neuron really *is* a linear regression by having one learn a straight line.

We make noisy data along **y = 2x + 1** and ask a one-neuron network to recover it.

In [ ]:
# noisy data along the line y = 2x + 1
X = np.random.uniform(-1, 1, size=(300, 1)).astype("float32")
y = (2 * X[:, 0] + 1 + np.random.normal(0, 0.12, size=300)).astype("float32")

plt.figure(figsize=(5.5, 4))
plt.scatter(X, y, s=10, alpha=0.4, color="crimson")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy data along y = 2x + 1")
plt.show()

Now we build the simplest possible network: **one neuron, one input, no activation** — which is
exactly linear regression. The Keras vocabulary:

> **`keras.Sequential`** — a network built as a simple *stack* of layers.
> **`layers.Dense(1)`** — a "fully-connected" layer with 1 neuron. *Dense* means every input
> connects to every neuron. With 1 neuron and no activation, it computes `w·x + b` — a line.
> **`layers.Input(shape=(1,))`** — tells the network to expect 1 input feature.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(1,)),     # one input feature (x)
    layers.Dense(1)               # one neuron, no activation -> a straight line
])
model.summary()

The summary shows the `Dense` layer has **2 parameters** — that's the weight (slope) and the bias
(intercept), the two numbers a line needs. The network will *learn* these from the data.

Next we **compile** it — tell it *how* to learn:

> **`model.compile(optimizer, loss)`** — sets up training.
> - **loss** = how we measure "wrong." For predicting numbers we use `"mse"` (mean squared error) —
>   the *same* squared-error idea as OLS regression!
> - **optimizer** = the strategy for improving the weights. `Adam` is the reliable default; the
>   `learning_rate` is how big a step it takes each update (we use a slightly larger 0.05 so this tiny
>   model learns quickly).

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05), loss="mse")
print("ready to train")

Finally we **fit** (train) it:

> **`model.fit(X, y, epochs=...)`** — runs the learning loop. One **epoch** = one full pass through
> all the training data. The network repeatedly: makes predictions → measures the loss → adjusts the
> weights to reduce it. We run 200 epochs. `verbose=0` just keeps it quiet.

In [ ]:
history = model.fit(X, y, epochs=200, verbose=0)

# read the learned weight and bias
w, b = model.layers[0].get_weights()
print(f"learned slope     (weight)   = {float(w[0,0]):.3f}   (true value: 2.0)")
print(f"learned intercept (bias)     = {float(b[0]):.3f}   (true value: 1.0)")

The single neuron recovered **slope ≈ 2** and **intercept ≈ 1** — it learned the line by
trial-and-error (gradient descent), not by a formula. This confirms it: **a neuron with no
activation is linear regression.** A neural network is just many of these stacked together, with
activations added. Let's see the fit and how the loss fell during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# the fitted line over the data
xs = np.linspace(-1, 1, 50).reshape(-1, 1)
axes[0].scatter(X, y, s=20, alpha=0.5, color="crimson")
axes[0].plot(xs, model.predict(xs, verbose=0), color="k", lw=2.5)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].set_title("The neuron learned the line")

# the loss curve: error shrinking over epochs
axes[1].plot(history.history["loss"], color="royalblue")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss (MSE)")
axes[1].set_yscale("log"); axes[1].set_title("Loss dropping as it learns")
plt.tight_layout(); plt.show()

In the loss curve (right) each epoch the error drops as the weights improve
that downward curve *is* learning. We'll watch this curve for every network.

### Task 1.1
Change the data to **y = −3x + 0.5** and re-run (recreate the model first so the weights reset).
Does the neuron recover slope ≈ −3 and intercept ≈ 0.5?

In [ ]:
# Your code here 👇


---
## 2.Activation functions

If a neuron is just a regression, why are neural networks so much more powerful? **The activation
function.**

Here's the key insight: if you stack linear neurons (no activation), you just get another line. Two
lines combined is still a line. No matter how many you stack, a network with no activations can only
draw straight lines. To bend and curve, we pass each neuron's output through a **non-linear**
function $f$.

The common activation functions:

- **ReLU** — `max(0, z)`: outputs 0 for negatives, passes positives through. Simple, fast, the
  default for hidden layers.
- **Sigmoid** — squashes any value into (0, 1): perfect for outputting a **probability**.
- **tanh** — squashes into (−1, 1): zero-centred, used inside sequence models.

Let's *see* why we need them. We'll try to fit a **curve**, y = sin(2x), with and without activations.

In [ ]:
# a curv relationship a straight line cannot fit
Xc = np.linspace(-3, 3, 400).reshape(-1, 1).astype("float32")
yc = np.sin(2 * Xc[:, 0]).astype("float32")

# (a) a linear network: one neuron, NO activation
linear_net = keras.Sequential([layers.Input((1,)), layers.Dense(1)])
linear_net.compile(optimizer="adam", loss="mse")
linear_net.fit(Xc, yc, epochs=200, verbose=0)

# (b) a network WITH hidden layers and ReLU activations
deep_net = keras.Sequential([
    layers.Input((1,)),
    layers.Dense(32, activation="relu"),   # hidden layer of 32 neurons
    layers.Dense(32, activation="relu"),   # another hidden layer
    layers.Dense(1)                        # output (linear, since we predict a number)
])
deep_net.compile(optimizer="adam", loss="mse")
deep_net.fit(Xc, yc, epochs=200, verbose=0)

plt.figure(figsize=(8, 4.5))
plt.scatter(Xc, yc, s=8, alpha=0.25, color="crimson", label="true curve sin(2x)")
plt.plot(Xc, linear_net.predict(Xc, verbose=0), color="#c96a2e", lw=2.5, label="no activation (fails)")
plt.plot(Xc, deep_net.predict(Xc, verbose=0), color="black", lw=2.5, label="with ReLU (works!)")
plt.legend(); plt.xlabel("x"); plt.ylabel("y")
plt.title("Activations are what let networks bend")
plt.show()

Without activations the

network can only draw a


straight line through a curve. The **black network**, with two ReLU hidden layers, traces the sine
wave almost perfectly. **This is the entire reason deep networks exist:** stacking non-linear layers
lets them approximate almost any relationship.

### Task 2.1
The `deep_net` has two hidden layers of 32 neurons. Build a *tiny* one (one hidden layer, just 3
neurons) and re-fit it on the same curve. Plot its prediction. Can 3 neurons capture the sine, or is
it too simple?

In [ ]:
# Your code here


---
## 3. From one neuron to a network

A single neuron is limited. The power comes from **stacking** them:

- **Input layer** — your features (one slot per input).
- **Hidden layers** — neurons in the middle. Each learns to detect some pattern; later layers combine
  simple patterns into complex ones. *"Deep" learning just means many hidden layers.*
- **Output layer** — produces the answer (one neuron for a single number or a yes/no probability;
  several for multi-class problems).

Two dials control a network's **capacity** (how complex a pattern it can learn):
- **width** — how many neurons per layer,
- **depth** — how many layers.

More capacity can learn more complex relationships — but, just like a high-degree polynomial or a
deep decision tree, too much capacity **overfits** (memorizes noise).

Here's the picture in words — information flows left to right, each arrow carrying a weight the
network learns:

```
   inputs        hidden layer      output
   (features)    (ReLU neurons)    (prediction)

     x1 ──┐      ┌──○──┐
     x2 ──┼──→   ├──○──┼──→  ○  ──→  ŷ
     x3 ──┘      └──○──┘
```

Every connection has a weight; training adjusts all of them together. A network with a few hidden
layers can have thousands of weights — which is why it needs more data than a simple regression.

---
## 4. How does a network actually learn?

We've said "it adjusts the weights to reduce the loss."

The process, repeated every epoch:
1. **Forward pass** — feed the data through the network, get predictions.
2. **Loss** — measure how wrong they are (e.g. MSE for numbers, cross-entropy for categories).
3. **Backpropagation** — work out how each weight contributed to the error (the "gradient" — which
   direction each weight should move to reduce the loss).
4. **Update** — nudge every weight a small step in the improving direction.

You don't implement any of this — Keras and the optimizer (`Adam`) handle it. But knowing the picture
explains every knob you'll touch: epochs (how many steps), learning rate (step size), loss (what
"downhill" means).

> **Try it yourself (highly recommended, ~10 min):** open **[playground.tensorflow.org](https://playground.tensorflow.org)**
> in your browser — a neural network you train with sliders, no code. Pick the spiral dataset, set
> activation to **Linear** and watch it fail; switch to **ReLU** and watch it succeed. Push the
> learning rate to 3 (it explodes) then 0.001 (it crawls). Add layers and neurons. Every concept from
> this notebook, made tangible.

---
## 5. A real network: predicting evapotranspiration

Let's build a network on a real problem. We'll predict daily **reference
evapotranspiration (ET₀)** — how much water the atmosphere can evaporate — from weather variables.
This is a *regression* with the DWD weather data.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dwd = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session3/Data/teaching_data/dwd_potsdam_daily.csv", parse_dates=["date"])

features = ["temperature_mean", "temperature_max", "temperature_min",
            "sunshine_hours", "humidity_mean"]
Xr = dwd[features].values.astype("float32")
yr = dwd["et0_mm"].values.astype("float32")
print("data:", Xr.shape, "->", len(features), "weather features per day")

**Two essential preparation steps for neural networks** :

1. **Train/test split** — as always, judge on unseen data.
2. **Scaling** — neural networks train far better when every feature is on a similar scale. We use `StandardScaler`, fit on the **training data
   only** so no information leaks from the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.3, random_state=42)

scaler = StandardScaler().fit(Xr_train)        # learn scaling from train only
Xr_train_s = scaler.transform(Xr_train)
Xr_test_s  = scaler.transform(Xr_test)
print("scaled and split:", Xr_train_s.shape, "train /", Xr_test_s.shape, "test")

Now build a small network: two hidden ReLU layers, and a single linear output neuron (because we
predict a number).

> **`validation_split=0.2`** in `fit` — holds back 20% of the *training* data to watch for
> overfitting *during* training (separate from the final test set).

In [ ]:
reg_net = keras.Sequential([
    layers.Input((len(features),)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)                          # linear output: a number (ET0)
])
reg_net.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = reg_net.fit(Xr_train_s, yr_train, epochs=60,
                      validation_split=0.2, verbose=0)
print("trained.")

In [ ]:
# watch the loss fall — training vs validation
plt.figure(figsize=(6.5, 4))
plt.plot(history.history["loss"], label="training loss", color="#3d6b4f")
plt.plot(history.history["val_loss"], label="validation loss", color="#c96a2e")
plt.xlabel("epoch"); plt.ylabel("loss (MSE)"); plt.legend()
plt.title("Training history — both curves fall and flatten (healthy)")
plt.show()

Both curves falling together and flattening = healthy training. Now the moment of truth — the
**predicted-vs-observed plot** on the test set (the same check we used in regression).

In [ ]:
pred = reg_net.predict(Xr_test_s, verbose=0).ravel()

from sklearn.metrics import r2_score
r2 = r2_score(yr_test, pred)

plt.figure(figsize=(5.5, 5.5))
plt.scatter(yr_test, pred, s=10, alpha=0.3, color="#143d59")
lims = [0, max(yr_test.max(), pred.max()) * 1.05]
plt.plot(lims, lims, "--", color="#c96a2e", lw=2, label="(1:1)")
plt.xlabel("actual ET0 [mm/d]"); plt.ylabel("predicted ET0 [mm/d]")
plt.xlim(lims); plt.ylim(lims); plt.legend()
plt.title(f"Neural network ET0 prediction (R² = {r2:.2f})")
plt.show()

The network predicts ET₀ well. You just trained a neural network on
real weather data, start to finish: build → compile → fit → check.

### Task 5.1
Add a third hidden layer (e.g. `layers.Dense(8, activation="relu")`) before the output, and retrain.
Does the test R² improve, stay the same, or get worse?

In [ ]:
# Your code here


---
## 6. A classification network: back to floods

Neural networks do classification too — and this lets us **return to the flood problem from Part 2**
and solve it with a network. Only two things change from the regression network:

- the **output layer** uses a **sigmoid** activation, so it outputs a *probability* of flooding (0–1),
- the **loss** becomes `"binary_crossentropy"` — the standard loss for yes/no problems.

Everything else — build, compile, fit, check — is identical. Let's rebuild the flood detector as a
neural net.

In [ ]:
# load the flood data + accumulated-rainfall features (as in the flood notebook)
ts = (pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session3/Data/teaching_data/timeseries_DE110260.csv", parse_dates=["date"])
        .sort_values("date").reset_index(drop=True))
thr = ts["discharge_spec_obs"].quantile(0.90)
ts["flood"] = (ts["discharge_spec_obs"] >= thr).astype(int)
for wdw in [3, 7, 14]:
    ts[f"precip_{wdw}day"] = ts["precipitation_mean"].rolling(wdw).sum()
ts = ts.dropna().reset_index(drop=True)

fl_features = ["precipitation_mean", "precip_3day", "precip_7day", "precip_14day", "temperature_mean"]
Xf = ts[fl_features].values.astype("float32")
yf = ts["flood"].values.astype("float32")

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    Xf, yf, test_size=0.3, random_state=42, stratify=yf)

fl_scaler = StandardScaler().fit(Xf_train)
Xf_train_s = fl_scaler.transform(Xf_train)
Xf_test_s  = fl_scaler.transform(Xf_test)
print("flood data ready:", Xf_train_s.shape)

Because floods are rare (10% of days), we again tell the network to take them seriously — the
neural-network version of `class_weight="balanced"` is a `class_weight` dictionary passed to `fit`.

In [ ]:
flood_net = keras.Sequential([
    layers.Input((len(fl_features),)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")       # sigmoid -> probability of flood
])
flood_net.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# weight the rare flood class up, so the network doesn't ignore it
n_no, n_yes = (yf_train == 0).sum(), (yf_train == 1).sum()
class_weight = {0: 1.0, 1: n_no / n_yes}

flood_net.fit(Xf_train_s, yf_train, epochs=40, validation_split=0.15,
              class_weight=class_weight, verbose=0)
print("flood network trained.")

In [ ]:
# evaluate the same way as the flood notebook: confusion matrix + recall
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, recall_score

# sigmoid gives probabilities; threshold at 0.5 to get yes/no
flood_prob = flood_net.predict(Xf_test_s, verbose=0).ravel()
flood_pred = (flood_prob >= 0.5).astype(int)

cm = confusion_matrix(yf_test, flood_pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(cm, display_labels=["not flood", "flood"]).plot(
    ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Neural network flood detector")
plt.tight_layout(); plt.show()

print(f"recall (floods caught): {recall_score(yf_test, flood_pred):.3f}")
print("Compare with the random forest from the flood notebook!")

The neural network catches floods comparably to the random forest from Part 2. Which leads to an
honest and important question…

## 7. When are neural networks actually worth it?

A crucial, well-documented lesson: **on small/medium *tabular* data (rows and columns of numbers,
like our catchment and weather tables), classical methods — random forests and gradient boosting —
usually match or beat neural networks, with far less tuning.** You saw it just now: the flood network
is *comparable* to the random forest, not dramatically better, and it needed scaling, class weights,
and more care.

So when *do* neural networks pull ahead? When the data is **large** and has **structure to exploit**:
- **images** (grids of pixels) — convolutional networks dominate,
- **text and language** — transformers (the technology behind ChatGPT),
- **sequences and time series** — recurrent networks and LSTMs.

That last one is the key for water science. The flood-*forecasting* cliffhanger from Part 2 — *will it
flood tomorrow?* — is a **sequence** problem, where a network that reads the *whole recent history* of
a catchment can shine where our hand-made features struggled. That's the natural next step beyond this
notebook (and the LSTM is exactly that kind of network).

> **The principle:** don't reach for the fanciest tool. Match the method to the data. Trees for
> tabular, neural networks for images / text / long sequences. Use a neural net because the *problem*
> calls for it, not because it sounds impressive.

---
## 8. Overfitting — the same lesson, one last time

Neural networks are flexible enough to **memorize noise** if you let them. The warning sign is the
same as always: the training loss keeps falling while the **validation loss turns back up**. Let's
force it to happen — a big network on a small, noisy dataset — then stop it.

In [ ]:
# small + noisy data, oversized network -> overfitting on purpose
Xo = np.linspace(-3, 3, 40).reshape(-1, 1).astype("float32")
yo = (np.sin(2 * Xo[:, 0]) + np.random.normal(0, 0.25, 40)).astype("float32")

big_net = keras.Sequential([
    layers.Input((1,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(1)
])
big_net.compile(optimizer="adam", loss="mse")
h = big_net.fit(Xo, yo, epochs=500, validation_split=0.3, verbose=0)

plt.figure(figsize=(6.5, 4))
plt.plot(h.history["loss"], label="training loss", color="#3d6b4f")
plt.plot(h.history["val_loss"], label="validation loss", color="#c96a2e")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Overfitting: validation loss rises while training loss keeps falling")
plt.show()

See the orange validation curve bottom out and then **climb** while the green training curve keeps
dropping — the network is now memorizing noise. The simplest cure is **EarlyStopping**: watch the
validation loss and stop when it stops improving, keeping the best weights.

> **`keras.callbacks.EarlyStopping`** — `patience=20` means "wait 20 epochs for improvement before
> stopping"; `restore_best_weights=True` rolls back to the best point. It's in almost every real
> training script.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True)

big_net2 = keras.Sequential([
    layers.Input((1,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(1)
])
big_net2.compile(optimizer="adam", loss="mse")
h2 = big_net2.fit(Xo, yo, epochs=500, validation_split=0.3,
                  callbacks=[early_stop], verbose=0)

print(f"stopped after {len(h2.history['loss'])} epochs (instead of 500)")
print("EarlyStopping kept the network from memorizing noise.")

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(h2.history["loss"], label="training loss", color="#3d6b4f")
plt.plot(h2.history["val_loss"], label="validation loss", color="#c96a2e")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Overfitting: validation loss rises while training loss keeps falling")
plt.show()

### Task 8.1
Another overfitting cure is **Dropout** — randomly switching off a fraction of neurons during each
training step, which forces the network not to over-rely on any one path. Add
`layers.Dropout(0.3)` after each Dense layer in `big_net` and retrain. Does the gap between training
and validation loss shrink?

In [ ]:
# Your code here


---
## Mini-exercise (20 min): build and tune your own network

Pick **one**:

**Option A — Regression:** predict ET₀ (as in Section 5), but design your own architecture. Try
different numbers of layers and neurons. Use `EarlyStopping`. Report the test R² and the
predicted-vs-observed plot. What architecture worked best?

**Option B — Classification:** build a flood detector (as in Section 6) with your own architecture and
`class_weight`. Report the confusion matrix and recall. Compare it to the random forest from the flood
notebook — does the network win, tie, or lose? (Remember the tabular-data lesson!)

For either: change **one thing at a time** — neurons, layers, learning rate
(`keras.optimizers.Adam(learning_rate=...)`), epochs — and note the effect. Write 3–4 sentences on
what you learned about how architecture choices change performance.

In [ ]:
# Your network here


---
---
## 🔑 Solutions / hints

<details><summary>Click to expand</summary>

**Task 1.1**
```python
y2 = (-3*X[:,0] + 0.5 + np.random.normal(0,0.12,300)).astype("float32")
m = keras.Sequential([layers.Input((1,)), layers.Dense(1)])
m.compile(optimizer=keras.optimizers.Adam(0.05), loss="mse")
m.fit(X, y2, epochs=200, verbose=0)
print(m.layers[0].get_weights())   # weight ~ -3, bias ~ 0.5
```

**Task 2.1** — 3 neurons underfit: the curve is captured only roughly (a few bends). More neurons =
smoother fit. This is capacity.

**Task 5.1** — adding a layer often barely changes the test R² on this easy problem, and can slightly
worsen it (more parameters, more overfitting risk). A good reminder that bigger isn't always better.

**Task 8.1**
```python
drop_net = keras.Sequential([
    layers.Input((1,)),
    layers.Dense(128, activation="relu"), layers.Dropout(0.3),
    layers.Dense(128, activation="relu"), layers.Dropout(0.3),
    layers.Dense(1)])
drop_net.compile(optimizer="adam", loss="mse")
hd = drop_net.fit(Xo, yo, epochs=500, validation_split=0.3, verbose=0)
# the train/validation gap shrinks -> less overfitting (though the fit is noisier).
```

**Mini-exercise** — there's no single right answer; grade on: sensible architecture, scaling,
train/test split, EarlyStopping, and an honest comparison. The expected insight for Option B: the
neural net is *competitive* with the random forest but rarely clearly better on this tabular data.
</details>

---
### Notes
- **TensorFlow / Keras**: `keras.Sequential` (stack of layers), `layers.Dense` (fully-connected
  layer), `layers.Input` (input shape), `model.compile` (set loss + optimizer), `model.fit` (train),
  `model.predict` (use), `EarlyStopping` (stop before overfitting), `Dropout` (overfitting cure).
- **Activations**: ReLU (hidden layers), sigmoid (probability output / binary classification), tanh
  (sequences).
- **Losses**: `mse` for regression, `binary_crossentropy` for yes/no classification.
- **Always**: scale features, split train/test, watch validation loss, prefer the simpler model when
  performance ties.
- **The honest principle**: neural networks shine on images, text, and sequences — not necessarily on
  small tabular data, where trees often win. Match the method to the problem.
- **The TensorFlow Playground** (playground.tensorflow.org) is the best way to build intuition.